# Лабораторна робота № 2
## Перевірка статистичної гіпотези про вигляд розподілу
### (критерії Колмогорова, χ² та пустих ящиків) і гіпотези однорідності (критерій Смирнова)

**Умова задачі:**  
Спостерігається вибірка $\bar{X} = (X_1, \ldots, X_n)$, де $\{X_i\}$ — незалежні однаково розподілені в.в., які мають **показниковий розподіл** з параметром $\lambda = 1$:
$$F(x) = 1 - e^{-x}, \quad x \geq 0$$

**Перетворення до рівномірного розподілу:**  
Якщо $\{\omega_i\}$ — незалежні рівномірно розподілені на $[0,1]$ в.в., то $X_i = -\ln(\omega_i)$.  
Для перевірки гіпотез $H_0: F = F_0$ за допомогою критеріїв Колмогорова, $\chi^2$ та пустих ящиків  
використовуємо перетворення $Y_i = F_0(X_i) = 1 - e^{-\lambda_0 X_i} \sim U[0,1]$ під $H_0$.

**Рівень значимості:** $\alpha = 0.05$  
**Розміри вибірки:** $n = 100, 1000, 10000$


In [ ]:
# ── Імпорти ────────────────────────────────────────────────────────────────
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Фіксуємо генератор для відтворюваності результатів
np.random.seed(42)

# ── Глобальні параметри ─────────────────────────────────────────────────────
ALPHA    = 0.05          # рівень значимості
NS       = [100, 1000, 10000]  # розміри вибірки
LAM_TRUE = 1.0           # справжній параметр розподілу (λ=1)
LAM_FAKE = 2.0           # «хибний» параметр для перевірки H₀: λ=2 (насправді λ=1)


## Генерація вибірки

Якщо $\omega \sim U[0,1]$, то $X = -\frac{\ln \omega}{\lambda} \sim \text{Exp}(\lambda)$.  
Перетворення $Y = 1 - e^{-\lambda_0 X}$ дає $Y \sim U[0,1]$ тоді і тільки тоді, коли $X \sim \text{Exp}(\lambda_0)$.


In [ ]:
def generate_exponential(n: int, lam: float = 1.0) -> np.ndarray:
    """
    Генерує вибірку розміру n з показникового розподілу Exp(lam).
    Метод інверсій: якщо ω ~ U[0,1], то X = -ln(ω)/λ ~ Exp(λ).
    """
    omega = np.random.uniform(0.0, 1.0, n)   # рівномірні псевдовипадкові числа
    return -np.log(omega) / lam               # зворотнє перетворення

def transform_to_uniform(X: np.ndarray, lam: float = 1.0) -> np.ndarray:
    """
    Перетворює вибірку X ~ Exp(lam) на рівномірний розподіл U[0,1].
    Y = F₀(X) = 1 − exp(−λ₀·X).
    Якщо X справді має розподіл Exp(λ₀), то Y ~ U[0,1].
    """
    return 1.0 - np.exp(-lam * X)

# Демонстрація: перетворена вибірка при вірній гіпотезі
X_demo = generate_exponential(1000, LAM_TRUE)
Y_demo = transform_to_uniform(X_demo, LAM_TRUE)
print(f"Середнє Y (очікується ~0.5):    {Y_demo.mean():.4f}")
print(f"Дисперсія Y (очікується ~1/12): {Y_demo.var():.4f}")


---
## Завдання 1. Критерій Колмогорова

**Статистика Колмогорова:**
$$D_n = \sqrt{n} \cdot \sup_x |F_n(x) - F_0(x)|$$
де $F_n(x)$ — емпірична функція розподілу, $F_0$ — гіпотетичний розподіл.

Після перетворення $Y_i = F_0(X_i)$ задача зводиться до перевірки $H_0: Y \sim U[0,1]$.

**Критична область:** $D_n > D_{1-\alpha}^{\text{Kolm}}$.  
Для $\alpha = 0.05$: $D_{0.95} \approx 1.36$.

Перевіряємо дві гіпотези:
- $H_0: \lambda = 1$ (насправді $\lambda = 1$) — очікуємо **не відхилити**  
- $H_0: \lambda = 1$ (насправді $\lambda = 2$) — очікуємо **відхилити**


In [ ]:
def kolmogorov_test(n: int, gen_lam: float, h0_lam: float) -> dict:
    """
    Критерій Колмогорова для перевірки H₀: X ~ Exp(h0_lam).
    
    Параметри:
        n       – розмір вибірки
        gen_lam – справжній параметр генерації
        h0_lam  – параметр гіпотетичного розподілу H₀
    
    Повертає словник з результатами.
    """
    # 1. Генеруємо вибірку зі справжнього розподілу
    X = generate_exponential(n, gen_lam)
    
    # 2. Перетворюємо до U[0,1] під H₀: Y_i = 1 − exp(−h0_lam · X_i)
    Y = transform_to_uniform(X, h0_lam)
    
    # 3. Обчислюємо статистику Колмогорова
    #    scipy.stats.kstest повертає D_n (ненормовану), перемножуємо на sqrt(n)
    D_raw, pval = stats.kstest(Y, 'uniform')
    Dn = np.sqrt(n) * D_raw          # нормована статистика
    
    # 4. Критичне значення з розподілу Колмогорова (рівень α)
    critical = stats.kstwobign.ppf(1 - ALPHA)   # ≈1.36 для α=0.05
    
    reject = Dn > critical           # рішення: відхилити H₀?
    
    return {
        "n": n, "gen_lam": gen_lam, "h0_lam": h0_lam,
        "Dn": Dn, "critical": critical, "pval": pval, "reject": reject
    }

# ── Друк результатів ────────────────────────────────────────────────────────
print("=" * 90)
print("ЗАВДАННЯ 1. КРИТЕРІЙ КОЛМОГОРОВА")
print("=" * 90)

for n in NS:
    print(f"\n{'─'*80}")
    print(f"  n = {n}")
    print(f"{'─'*80}")

    # Гіпотеза вірна (H₀: λ=1, насправді λ=1)
    r = kolmogorov_test(n, gen_lam=LAM_TRUE, h0_lam=LAM_TRUE)
    verdict = "ВІДХИЛИТИ H₀ ✗" if r['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  H₀: λ=1 (насправді λ=1) | Dₙ={r['Dn']:.5f}  крит.={r['critical']:.4f}  "
          f"p-знач.={r['pval']:.4f}  → {verdict}")
    
    # Гіпотеза хибна (H₀: λ=1, насправді λ=2)
    r = kolmogorov_test(n, gen_lam=LAM_FAKE, h0_lam=LAM_TRUE)
    verdict = "ВІДХИЛИТИ H₀ ✗" if r['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  H₀: λ=1 (насправді λ=2) | Dₙ={r['Dn']:.5f}  крит.={r['critical']:.4f}  "
          f"p-знач.={r['pval']:.4f}  → {verdict}")


In [ ]:
# ── Візуалізація: ECDF vs теоретичний CDF ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, n in zip(axes, NS):
    # Вибірка при вірній H₀
    Y_true = transform_to_uniform(generate_exponential(n, LAM_TRUE), LAM_TRUE)
    Y_fake = transform_to_uniform(generate_exponential(n, LAM_FAKE), LAM_TRUE)
    
    x_grid = np.linspace(0, 1, 500)
    
    # Емпіричний CDF (вірна H₀)
    Y_s = np.sort(Y_true)
    ax.step(Y_s, np.arange(1, n+1)/n, where='post', color='blue', lw=1.5,
            label='ECDF (H₀ вірна)')
    # Емпіричний CDF (хибна H₀)
    Y_s2 = np.sort(Y_fake)
    ax.step(Y_s2, np.arange(1, n+1)/n, where='post', color='red', lw=1.5,
            ls='--', label='ECDF (H₀ хибна)')
    # Теоретичний U[0,1]
    ax.plot(x_grid, x_grid, 'g-', lw=2, label='F₀ = U[0,1]')
    
    ax.set_title(f'n = {n}', fontsize=11)
    ax.set_xlabel('y'); ax.set_ylabel('F(y)')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Критерій Колмогорова: ECDF vs теоретичний CDF', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Завдання 2. Критерій $\chi^2$ (Пірсона)

Ділимо $[0,1]$ на $k$ рівних проміжків. Кількість спостережень у кожному проміжку:
$$n_j = \#\{i : Y_i \in [(j-1)/k, \, j/k)\}$$

**Статистика:**
$$\chi^2 = \sum_{j=1}^{k} \frac{(n_j - n/k)^2}{n/k} = \frac{k}{n}\sum_{j=1}^{k}(n_j - n/k)^2$$

**Критична область:** $\chi^2 > \chi^2_{1-\alpha}(k-1)$.  
**Вибір $k$:** з умови $n/k \geq 10$, тобто $k = \lfloor n/10 \rfloor$.


In [ ]:
def chisq_test(n: int, gen_lam: float, h0_lam: float) -> dict:
    """
    Критерій χ² для перевірки рівномірності трансформованої вибірки.
    
    k = n//10 (умова: n/k ≥ 10, тобто очікувана кількість в кожній клітинці ≥ 10)
    """
    # Кількість проміжків: k ≤ n/10
    k = max(2, n // 10)
    
    # 1. Генерація та перетворення
    X = generate_exponential(n, gen_lam)
    Y = transform_to_uniform(X, h0_lam)  # Y ~ U[0,1] під H₀
    
    # 2. Підрахунок кількостей у кожному проміжку [j/k, (j+1)/k)
    counts, _ = np.histogram(Y, bins=k, range=(0.0, 1.0))
    
    # 3. Очікувана кількість у кожному проміжку (рівномірна → n/k)
    expected = n / k
    
    # 4. Статистика χ²
    chi2_stat = float(np.sum((counts - expected) ** 2 / expected))
    
    # 5. Критичне значення χ²_{1-α}(k-1)
    critical = stats.chi2.ppf(1 - ALPHA, df=k - 1)
    
    reject = chi2_stat > critical
    
    return {
        "n": n, "k": k, "gen_lam": gen_lam, "h0_lam": h0_lam,
        "chi2": chi2_stat, "critical": critical, "reject": reject
    }

print("=" * 90)
print("ЗАВДАННЯ 2. КРИТЕРІЙ χ²")
print("=" * 90)

for n in NS:
    print(f"\n{'─'*80}")
    r0 = chisq_test(n, LAM_TRUE, LAM_TRUE)
    r1 = chisq_test(n, LAM_FAKE, LAM_TRUE)
    print(f"  n = {n},  k = {r0['k']}  (очікувана кількість в клітинці = {n/r0['k']:.0f})")
    v0 = "ВІДХИЛИТИ H₀ ✗" if r0['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    v1 = "ВІДХИЛИТИ H₀ ✗" if r1['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  H₀: λ=1 (насправді λ=1) | χ²={r0['chi2']:.4f}  крит.={r0['critical']:.4f}  → {v0}")
    print(f"  H₀: λ=1 (насправді λ=2) | χ²={r1['chi2']:.4f}  крит.={r1['critical']:.4f}  → {v1}")


In [ ]:
# ── Візуалізація: гістограми ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, n in enumerate(NS):
    k = max(2, n // 10)
    expected = n / k
    bins_edges = np.linspace(0, 1, k + 1)

    for row, (gen_lam, label) in enumerate([(LAM_TRUE, "H₀ вірна (λ=1)"),
                                             (LAM_FAKE, "H₀ хибна (λ=2)")]):
        ax = axes[row][col]
        X = generate_exponential(n, gen_lam)
        Y = transform_to_uniform(X, LAM_TRUE)
        counts, _ = np.histogram(Y, bins=k, range=(0, 1))

        ax.bar(bins_edges[:-1], counts, width=1/k, align='edge',
               color='steelblue' if row == 0 else 'salmon',
               edgecolor='black', lw=0.5, alpha=0.8, label='Спостережені')
        ax.axhline(expected, color='green', lw=2, ls='--', label=f'Очікувані = {expected:.0f}')
        ax.set_title(f'n={n}, {label}', fontsize=9)
        ax.set_xlabel('y'); ax.set_ylabel('кількість')
        ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Критерій χ²: спостережені vs очікувані частоти', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Завдання 3. Критерій пустих ящиків

Ділимо $[0,1]$ на $m$ рівних ящиків і кидаємо $n$ куль рівномірно.  
$V_0$ — кількість **порожніх** ящиків.

**Математичне сподівання та дисперсія під $H_0$:**
$$\mathbb{E}[V_0] = m\left(1 - \frac{1}{m}\right)^n$$
$$\mathbb{D}[V_0] = m\left(1-\tfrac{1}{m}\right)^n\left(1-\left(1-\tfrac{1}{m}\right)^n\right)
    + m(m-1)\left(1-\tfrac{2}{m}\right)^n - m(m-1)\left(1-\tfrac{1}{m}\right)^{2n}$$

**Стандартизована статистика:** $Z = \dfrac{V_0 - \mathbb{E}[V_0]}{\sqrt{\mathbb{D}[V_0]}} \xrightarrow{d} N(0,1)$

**Критична область:** $|Z| > z_{1-\alpha/2}$.  
**Вибір $m$:** з умови $n > m \ln m$, тобто $m = \lfloor n / \ln n \rfloor$.


In [ ]:
def empty_boxes_test(n: int, gen_lam: float, h0_lam: float) -> dict:
    """
    Критерій пустих ящиків для перевірки рівномірності.
    
    m = floor(n / ln(n))  — вибір кількості ящиків
    V₀ — кількість порожніх ящиків
    Асимптотика: Z = (V₀ − E[V₀]) / sqrt(D[V₀]) → N(0,1)
    """
    # Кількість ящиків: з умови n > m*ln(m), беремо m ≈ n/ln(n)
    m = max(2, int(n / np.log(n)))
    
    # 1. Генерація та перетворення до U[0,1] під H₀
    X = generate_exponential(n, gen_lam)
    Y = transform_to_uniform(X, h0_lam)
    
    # 2. Розподіляємо спостереження по m ящиках [0,1/m), [1/m,2/m), ...
    box_ids = np.clip(np.floor(Y * m).astype(int), 0, m - 1)
    
    # 3. Кількість пустих ящиків
    V0 = m - len(np.unique(box_ids))
    
    # 4. Математичне сподівання та дисперсія V₀ під H₀
    p = 1.0 / m                          # ймовірність потрапити в один ящик
    EV0 = m * (1 - p) ** n               # очікувана кількість пустих ящиків
    DV0 = (m * (1 - p)**n * (1 - (1 - p)**n)        # дисперсія (формула з лекції)
           + m * (m - 1) * (1 - 2*p)**n
           - m * (m - 1) * (1 - p)**(2*n))
    DV0 = max(DV0, 1e-10)                # захист від нульової дисперсії
    
    # 5. Стандартизована статистика
    Z = (V0 - EV0) / np.sqrt(DV0)
    
    # 6. Критичне значення N(0,1) для двостороннього критерію
    critical = stats.norm.ppf(1 - ALPHA / 2)    # ≈ 1.96 для α=0.05
    
    reject = abs(Z) > critical
    
    return {
        "n": n, "m": m, "V0": V0, "EV0": EV0, "DV0": DV0,
        "Z": Z, "critical": critical, "reject": reject
    }

print("=" * 90)
print("ЗАВДАННЯ 3. КРИТЕРІЙ ПУСТИХ ЯЩИКІВ")
print("=" * 90)

for n in NS:
    print(f"\n{'─'*80}")
    r0 = empty_boxes_test(n, LAM_TRUE, LAM_TRUE)
    r1 = empty_boxes_test(n, LAM_FAKE, LAM_TRUE)
    print(f"  n = {n},  m = {r0['m']}")
    v0 = "ВІДХИЛИТИ H₀ ✗" if r0['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    v1 = "ВІДХИЛИТИ H₀ ✗" if r1['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  H₀: λ=1 (насправді λ=1) | V₀={r0['V0']}  E[V₀]={r0['EV0']:.2f}  "
          f"Z={r0['Z']:.4f}  крит.=±{r0['critical']:.4f}  → {v0}")
    print(f"  H₀: λ=1 (насправді λ=2) | V₀={r1['V0']}  E[V₀]={r1['EV0']:.2f}  "
          f"Z={r1['Z']:.4f}  крит.=±{r1['critical']:.4f}  → {v1}")


---
## Завдання 4. Критерій однорідності Смирнова

Маємо дві вибірки $X_1,\ldots,X_m$ та $Y_1,\ldots,Y_n$.  

**Статистика Смирнова:**
$$D_{m,n} = \sup_x |F_m(x) - G_n(x)|$$

**Нормована статистика:**
$$T = D_{m,n} \cdot \sqrt{\frac{mn}{m+n}} \xrightarrow{d} \text{Kolmogorov}$$

**Критична область:** $T > D_{1-\alpha}^{\text{Kolm}} \approx 1.36$ при $\alpha = 0.05$.

Обираємо $m = n$.

Перевіряємо:
- $H_0$: дві вибірки мають однаковий розподіл Exp(1) — очікуємо **не відхилити**  
- $H_0$: однорідність, коли перша Exp(1), друга Exp(2) — очікуємо **відхилити**


In [ ]:
def smirnov_test(n: int, lam1: float, lam2: float) -> dict:
    """
    Критерій однорідності Смирнова для двох вибірок розміру n.
    
    Статистика: T = D_{n,n} * sqrt(n/2) → розподіл Колмогорова
    """
    m = n   # за умовою задачі обираємо рівні розміри вибірок
    
    # 1. Генеруємо дві незалежні вибірки
    X = generate_exponential(m, lam1)
    Y = generate_exponential(n, lam2)
    
    # 2. Двовибірковий тест Колмогорова-Смирнова (scipy)
    D_raw, pval = stats.ks_2samp(X, Y)
    
    # 3. Нормована статистика
    T = D_raw * np.sqrt(m * n / (m + n))
    
    # 4. Критичне значення (той самий розподіл Колмогорова)
    critical = stats.kstwobign.ppf(1 - ALPHA)
    
    reject = T > critical
    
    return {
        "n": n, "lam1": lam1, "lam2": lam2,
        "D": D_raw, "T": T, "critical": critical, "pval": pval, "reject": reject
    }

print("=" * 90)
print("ЗАВДАННЯ 4. КРИТЕРІЙ ОДНОРІДНОСТІ СМИРНОВА")
print("=" * 90)

for n in NS:
    print(f"\n{'─'*80}")
    r0 = smirnov_test(n, LAM_TRUE, LAM_TRUE)
    r1 = smirnov_test(n, LAM_TRUE, LAM_FAKE)
    v0 = "ВІДХИЛИТИ H₀ ✗" if r0['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    v1 = "ВІДХИЛИТИ H₀ ✗" if r1['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  n = {n}")
    print(f"  H₀: однорід. (X~Exp(1), Y~Exp(1)) | T={r0['T']:.5f}  крит.={r0['critical']:.4f}  "
          f"p={r0['pval']:.4f}  → {v0}")
    print(f"  H₀: однорід. (X~Exp(1), Y~Exp(2)) | T={r1['T']:.5f}  крит.={r1['critical']:.4f}  "
          f"p={r1['pval']:.4f}  → {v1}")


In [ ]:
# ── Зведений графік: нормовані статистики для всіх завдань ─────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
titles    = ['Колмогорова', 'χ²  (відн.)', 'Пустих ящиків', 'Смирнова']
colors    = [['#2196F3', '#F44336'], ['#4CAF50', '#FF9800'],
             ['#9C27B0', '#E91E63'], ['#00BCD4', '#795548']]

for col, (test_fn, ax, title, clr) in enumerate(zip(
        [kolmogorov_test, chisq_test, empty_boxes_test, smirnov_test],
        axes, titles, colors)):
    
    stats_true, stats_false = [], []
    for n in NS:
        if col < 3:   # одновибіркові тести
            r0 = test_fn(n, LAM_TRUE, LAM_TRUE)
            r1 = test_fn(n, LAM_FAKE, LAM_TRUE)
            s0 = r0.get('Dn', r0.get('chi2', r0.get('Z', 0)))
            s1 = r1.get('Dn', r1.get('chi2', r1.get('Z', 0)))
            crit = r0.get('critical', 1.96)
        else:          # двовибірковий тест Смирнова
            r0 = test_fn(n, LAM_TRUE, LAM_TRUE)
            r1 = test_fn(n, LAM_TRUE, LAM_FAKE)
            s0, s1, crit = r0['T'], r1['T'], r0['critical']
        
        # Для χ² нормуємо відносно критичного значення
        if col == 1:
            s0 /= crit; s1 /= crit; crit = 1.0
        
        stats_true.append(abs(s0)); stats_false.append(abs(s1))
    
    x = np.arange(len(NS))
    width = 0.35
    ax.bar(x - width/2, stats_true,  width, label='H₀ вірна', color=clr[0], alpha=0.8)
    ax.bar(x + width/2, stats_false, width, label='H₀ хибна', color=clr[1], alpha=0.8)
    ax.axhline(crit, color='black', ls='--', lw=1.5, label=f'крит.={crit:.2f}')
    ax.set_xticks(x); ax.set_xticklabels([f'n={n}' for n in NS], fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Лабораторна 2 — Значення статистик по завданнях', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Висновки

| Завдання | Критерій | n=100 | n=1000 | n=10000 |
|---|---|---|---|---|
| 1 | Колмогорова | ✓ / ✗ | ✓ / ✗ | ✓ / ✗ |
| 2 | χ² Пірсона | ✓ / ✗ | ✓ / ✗ | ✓ / ✗ |
| 3 | Пустих ящиків | ✓ / ✗ | ✓ / ✗ | ✓ / ✗ |
| 4 | Смирнова | ✓ / ✗ | ✓ / ✗ | ✓ / ✗ |

(*✓ = не відхилено H₀, ✗ = H₀ відхилено*)

- **Із збільшенням n** потужність усіх критеріїв зростає: хибна H₀ відхиляється надійніше.  
- **Критерій Колмогорова** є найбільш потужним для неперервних розподілів.  
- **Критерій χ²** залежить від вибору k; при малому n чутливість нижча.  
- **Критерій Смирнова** є двовибірковим аналогом критерію Колмогорова.
